    # Study B Controllability Analysis

    This notebook is the dedicated controllability analysis for Study B.

    It focuses on:
    1. the study primary controllability metric (`CHR / Controlled Hallucination Rate`)
    2. metric-level control performance across the study metrics
    3. study-level aggregate controllability
    4. benchmark-level controllability context where the summary files are available

    ## Controlled metric profile

    The grouped profile rows are expected to show:
    - the controlled value for each metric
    - the baseline value when available
    - the baseline delta
    - the metric-level `control_score`
    - threshold provenance (`formal`, `provisional`, `derived`)
    - whether a metric is active or reporting-only

    Study metrics tracked here:
    - `sycophancy_probability`
- `evidence_hallucination`
- `flip_rate`
- `controlled_accuracy`


In [ ]:
import sys
from pathlib import Path

for candidate_root in [Path.cwd(), Path.cwd().parent]:
    src_dir = candidate_root / "src"
    if src_dir.exists() and str(src_dir.resolve()) not in sys.path:
        sys.path.insert(0, str(src_dir.resolve()))

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

RESULTS_DIR = next(
    (p for p in [Path("results"), Path("../results"), Path("../../results")] if p.exists()),
    Path("results"),
)
print(f"Using RESULTS_DIR: {RESULTS_DIR.resolve()}")


In [ ]:
STUDY_CODE = "B"
PRIMARY_LABEL = "CHR / Controlled Hallucination Rate"


In [ ]:
def load_controllability_study_payload(study_code: str):
    study_file_map = {
        "A": "ctrl_study_a_results.json",
        "B": "ctrl_study_b_results.json",
        "C": "ctrl_study_c_results.json",
    }
    target_file = study_file_map[study_code]

    primary_rows = []
    profile_rows = []
    summary_rows = []

    if not RESULTS_DIR.exists():
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    for model_dir in sorted(path for path in RESULTS_DIR.iterdir() if path.is_dir()):
        study_path = model_dir / target_file
        if study_path.exists():
            payload = json.loads(study_path.read_text(encoding="utf-8"))
            primary_metric = payload.get("primary_metric", {})
            aggregate = payload.get("aggregate", {})
            primary_rows.append(
                {
                    "model": payload.get("model", model_dir.name),
                    "study": payload.get("study", study_code),
                    "primary_metric_name": primary_metric.get("metric_name"),
                    "primary_metric_value": primary_metric.get("value"),
                    "primary_ci_lower": primary_metric.get("ci_lower"),
                    "primary_ci_upper": primary_metric.get("ci_upper"),
                    "primary_threshold_value": (primary_metric.get("threshold") or {}).get("threshold_value"),
                    "primary_threshold_status": (primary_metric.get("threshold") or {}).get("status"),
                    "aggregate_score": aggregate.get("score"),
                    "aggregate_weighting_policy": aggregate.get("weighting_policy"),
                    "aggregate_provisional_components": aggregate.get("provisional_components"),
                }
            )

            for entry in payload.get("controlled_profile", []):
                threshold = entry.get("threshold") or {}
                profile_rows.append(
                    {
                        "model": payload.get("model", model_dir.name),
                        "study": payload.get("study", study_code),
                        "metric_name": entry.get("metric_name"),
                        "classification": entry.get("classification"),
                        "controlled_value": entry.get("controlled_value"),
                        "baseline_value": entry.get("baseline_value"),
                        "delta_from_baseline": entry.get("delta_from_baseline"),
                        "compliance_anchor": entry.get("compliance_anchor"),
                        "outcome_gain": entry.get("outcome_gain"),
                        "control_score": entry.get("control_score"),
                        "included_in_rollup": entry.get("included_in_rollup"),
                        "threshold_value": threshold.get("threshold_value"),
                        "threshold_status": threshold.get("status"),
                        "threshold_enforcement_mode": threshold.get("enforcement_mode"),
                        "threshold_source": threshold.get("threshold_source"),
                        "meets_threshold": threshold.get("meets_threshold"),
                        "notes": entry.get("notes"),
                    }
                )

        summary_path = model_dir / "controllability_summary.json"
        if summary_path.exists():
            summary_payload = json.loads(summary_path.read_text(encoding="utf-8"))
            studies = summary_payload.get("studies", {})
            study_payload = studies.get(study_code)
            benchmark_payload = summary_payload.get("benchmark_control", {})
            if study_payload:
                summary_rows.append(
                    {
                        "model": summary_payload.get("model", model_dir.name),
                        "study": study_code,
                        "study_score": (study_payload.get("aggregate") or {}).get("score"),
                        "benchmark_control_score": benchmark_payload.get("score"),
                        "benchmark_provisional_components": benchmark_payload.get("provisional_components"),
                    }
                )

    return pd.DataFrame(primary_rows), pd.DataFrame(profile_rows), pd.DataFrame(summary_rows)


## Ranking


In [ ]:
primary_df, profile_df, summary_df = load_controllability_study_payload(STUDY_CODE)

if primary_df.empty:
    print(f"No controllability results found yet for Study {STUDY_CODE}.")
else:
    ranking_df = primary_df.sort_values("primary_metric_value", ascending=False).reset_index(drop=True)
    ranking_df["rank"] = range(1, len(ranking_df) + 1)
    ranking_df = ranking_df[
        [
            "rank",
            "model",
            "primary_metric_name",
            "primary_metric_value",
            "primary_ci_lower",
            "primary_ci_upper",
            "aggregate_score",
            "aggregate_provisional_components",
        ]
    ]
    print(f"Study {STUDY_CODE} controllability ranking")
    print("=" * 100)
    display(ranking_df)


## Primary Metric


In [ ]:
if primary_df.empty:
    print("Skipping primary metric plot - no controllability study results found.")
else:
    plot_df = primary_df.sort_values("primary_metric_value", ascending=False).reset_index(drop=True)
    vals = pd.to_numeric(plot_df["primary_metric_value"], errors="coerce").fillna(0.0).values
    ci_low = pd.to_numeric(plot_df["primary_ci_lower"], errors="coerce").fillna(plot_df["primary_metric_value"]).values
    ci_high = pd.to_numeric(plot_df["primary_ci_upper"], errors="coerce").fillna(plot_df["primary_metric_value"]).values
    yerr = np.vstack([vals - ci_low, ci_high - vals])

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.bar(plot_df["model"], vals, yerr=yerr, capsize=6, alpha=0.8, color="#4C72B0")
    ax.set_title(f"Study {STUDY_CODE} Primary Controllability Metric", fontsize=14, fontweight="bold")
    ax.set_ylabel("Primary controllability value")
    ax.set_xlabel("Model")
    plt.xticks(rotation=45, ha="right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


## Metric-Level Control Scores


In [ ]:
if profile_df.empty:
    print("Skipping control-score profile plot - no controlled metric profile rows found.")
else:
    score_df = profile_df.dropna(subset=["control_score"]).copy()
    if score_df.empty:
        print("No metric-level control scores are populated yet.")
    else:
        fig, ax = plt.subplots(figsize=(16, 7))
        sns.barplot(
            data=score_df,
            x="metric_name",
            y="control_score",
            hue="model",
            ax=ax,
        )
        ax.set_title(f"Study {STUDY_CODE} Metric-Level Control Scores", fontsize=14, fontweight="bold")
        ax.set_xlabel("Metric")
        ax.set_ylabel("Control score")
        plt.xticks(rotation=35, ha="right")
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()


## Controlled vs Baseline Shift


In [ ]:
if profile_df.empty:
    print("Skipping baseline-delta plot - no controlled metric profile rows found.")
else:
    delta_df = profile_df.dropna(subset=["delta_from_baseline"]).copy()
    if delta_df.empty:
        print("No baseline deltas are populated yet.")
    else:
        fig, ax = plt.subplots(figsize=(16, 7))
        sns.barplot(
            data=delta_df,
            x="metric_name",
            y="delta_from_baseline",
            hue="model",
            ax=ax,
        )
        ax.axhline(0.0, color="black", linewidth=1, alpha=0.4)
        ax.set_title(f"Study {STUDY_CODE} Controlled Metric Shift vs Baseline", fontsize=14, fontweight="bold")
        ax.set_xlabel("Metric")
        ax.set_ylabel("Controlled minus baseline")
        plt.xticks(rotation=35, ha="right")
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()


## Study Aggregate


In [ ]:
if primary_df.empty:
    print("Skipping study aggregate plot - no controllability study results found.")
else:
    agg_df = primary_df.dropna(subset=["aggregate_score"]).sort_values("aggregate_score", ascending=False)
    if agg_df.empty:
        print("No study-level controllability aggregate scores are available yet.")
    else:
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.bar(agg_df["model"], agg_df["aggregate_score"], color="#55A868", alpha=0.85)
        ax.set_title(f"Study {STUDY_CODE} Aggregate Controllability", fontsize=14, fontweight="bold")
        ax.set_xlabel("Model")
        ax.set_ylabel("Aggregate control score")
        plt.xticks(rotation=45, ha="right")
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()


## Threshold Provenance


In [ ]:
if profile_df.empty:
    print("Skipping threshold/provenance table - no controlled profile rows found.")
else:
    threshold_cols = [
        "model",
        "metric_name",
        "classification",
        "control_score",
        "threshold_value",
        "threshold_status",
        "threshold_enforcement_mode",
        "threshold_source",
        "meets_threshold",
    ]
    display(profile_df[threshold_cols].sort_values(["metric_name", "model"]).reset_index(drop=True))


## Overall Benchmark Controllability


In [ ]:
if summary_df.empty:
    print("No benchmark-level controllability summary files were found yet.")
else:
    overall_df = summary_df.sort_values("benchmark_control_score", ascending=False).reset_index(drop=True)
    display(overall_df)

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(overall_df["model"], overall_df["benchmark_control_score"], color="#C44E52", alpha=0.85)
    ax.set_title("Overall Benchmark Controllability", fontsize=14, fontweight="bold")
    ax.set_xlabel("Model")
    ax.set_ylabel("Benchmark control score")
    plt.xticks(rotation=45, ha="right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
